# Create BioASQ retrieval subsets

This notebook combines the expert-authored BioASQ 11b questions and gold PubMed document annotations with a corpus of PubMed titles and abstracts. It can create separate retrieval samples by question type and by whether a question has one or multiple relevant documents.

Only questions whose complete gold-document set is present in the corpus are eligible. This avoids silently converting a multi-document question into a single-document question.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/retrieval-benchlab")
if "google.colab" in sys.modules:
    if not (REPO_ROOT / ".git").exists():
        !git clone --depth 1 https://github.com/lohex/retrieval-benchlab.git {REPO_ROOT}
    %cd {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

!pip -q install -U datasets tqdm


In [ ]:
import logging
from collections import Counter

import numpy as np

from src.evaluate import register_dataset
from src.io import (
    load_bioasq_benchmark,
    mount_google_drive,
    sample_directory,
    save_calibration_set,
    save_sample,
)
from src.sampling import (
    add_random_negative_documents,
    sample_calibration_documents,
    sample_queries_and_positives,
    validate_sample,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
logger = logging.getLogger("bioasq-sample")


## Configuration

`N_QUERIES_PER_SUBSET = 90` keeps all six subsets equally sized. Set it to `None` to use every eligible question in each subset.

In [ ]:
CORPUS_DATASET_NAME = "DinoStackAI/bioasq-rag-13b-resplit"
CORPUS_CONFIG = "corpus"
CORPUS_SPLIT = "train"
QUESTIONS_SOURCE = "https://zenodo.org/api/records/7655130/files/training11b.json/content"

N_QUERIES_PER_SUBSET = 90
N_CORPUS_DOCS = 30_000
N_CALIBRATION_DOCS = 5_000
SEED = 42
CALIBRATION_SEED = 43

OUTPUT_ROOT = "/content/drive/MyDrive/Retreaval/data"
CALIBRATION_OUTPUT_DIR = (
    "/content/drive/MyDrive/Retreaval/calibration/bioasq-5k"
)
CALIBRATION_QUESTION_TYPES = ("list", "factoid", "summary")


## Load the shared benchmark once

The source corpus is loaded only once and reused by the calibration and subset-creation blocks. Empty source documents are discarded while loading.

In [ ]:
mount_google_drive()
benchmark = load_bioasq_benchmark(
    corpus_dataset_name=CORPUS_DATASET_NAME,
    corpus_config=CORPUS_CONFIG,
    corpus_split=CORPUS_SPLIT,
    questions_source=QUESTIONS_SOURCE,
)


## Create the shared calibration set

The calibration set contains 5,000 documents that are not relevant to any complete `list`, `factoid`, or `summary` question. It is stored outside the retrieval-dataset directory and excluded from every sampled evaluation corpus.

In [ ]:
def create_bioasq_calibration_set(
    n_documents=N_CALIBRATION_DOCS,
    seed=CALIBRATION_SEED,
    output_dir=CALIBRATION_OUTPUT_DIR,
    benchmark=None,
):
    """Create and persist the shared BioASQ calibration set."""
    if benchmark is None:
        benchmark = load_bioasq_benchmark(
            corpus_dataset_name=CORPUS_DATASET_NAME,
            corpus_config=CORPUS_CONFIG,
            corpus_split=CORPUS_SPLIT,
            questions_source=QUESTIONS_SOURCE,
        )

    (
        _,
        source_relevant_docs,
        source_query_types,
        source_corpus,
        source_metadata,
    ) = benchmark
    calibration_corpus = sample_calibration_documents(
        source_corpus=source_corpus,
        relevant_docs=source_relevant_docs,
        query_types=source_query_types,
        n_documents=n_documents,
        rng=np.random.default_rng(seed),
        protected_question_types=CALIBRATION_QUESTION_TYPES,
    )
    protected_relevant_ids = set().union(
        *(
            source_relevant_docs[query_id]
            for query_id, question_type in source_query_types.items()
            if question_type in CALIBRATION_QUESTION_TYPES
        )
    )
    metadata = {
        **source_metadata,
        "set_name": "bioasq-5k",
        "purpose": "embedding calibration",
        "n_documents": len(calibration_corpus),
        "seed": seed,
        "protected_question_types": list(CALIBRATION_QUESTION_TYPES),
        "n_protected_relevant_documents": len(protected_relevant_ids),
        "sampling": "uniform sample excluding protected relevant documents",
    }
    save_calibration_set(output_dir, calibration_corpus, metadata)
    return calibration_corpus, metadata, Path(output_dir)


(
    calibration_corpus,
    calibration_metadata,
    calibration_output_dir,
) = create_bioasq_calibration_set(benchmark=benchmark)
calibration_document_ids = frozenset(calibration_corpus)
print(
    f"Created calibration set: {calibration_output_dir} "
    f"({len(calibration_document_ids)} documents)"
)


## Final creation function

`create_bioasq_sample` filters, samples, validates, stores metadata, replaces only the named subset directory, and registers the saved dataset version. The accepted `documents` values are `"all"`, `"one"`, and `"multiple"`.

In [ ]:
def create_bioasq_sample(
    question_type=None,
    documents="all",
    subset_name=None,
    n_queries=N_QUERIES_PER_SUBSET,
    n_corpus_docs=N_CORPUS_DOCS,
    seed=SEED,
    output_root=OUTPUT_ROOT,
    benchmark=None,
    calibration_document_ids=None,
    calibration_set_path=None,
):
    """Create, validate, persist, register, and return one BioASQ sample."""
    if n_corpus_docs <= 0:
        raise ValueError("n_corpus_docs must be positive")
    question_type = question_type.lower() if question_type else None
    documents = documents.lower()

    if benchmark is None:
        benchmark = load_bioasq_benchmark(
            corpus_dataset_name=CORPUS_DATASET_NAME,
            corpus_config=CORPUS_CONFIG,
            corpus_split=CORPUS_SPLIT,
            questions_source=QUESTIONS_SOURCE,
        )

    (
        source_queries,
        source_relevant_docs,
        source_query_types,
        source_corpus,
        source_metadata,
    ) = benchmark
    rng = np.random.default_rng(seed)

    logger.info(
        "Selecting questions for type=%s and documents=%s",
        question_type or "all",
        documents,
    )
    queries, relevant_docs, query_types, n_eligible = sample_queries_and_positives(
        queries=source_queries,
        relevant_docs=source_relevant_docs,
        query_types=source_query_types,
        n_queries=n_queries,
        rng=rng,
        question_type=question_type,
        documents=documents,
    )

    logger.info("Sampling random negative documents")
    corpus = add_random_negative_documents(
        source_corpus=source_corpus,
        relevant_docs=relevant_docs,
        target_size=n_corpus_docs,
        rng=rng,
        excluded_document_ids=calibration_document_ids,
    )
    validate_sample(
        queries,
        relevant_docs,
        corpus,
        expected_queries=len(queries),
        expected_corpus_docs=n_corpus_docs,
    )

    type_label = question_type or "all"
    if subset_name is None:
        subset_name = "current" if type_label == documents == "all" else f"{type_label}-{documents}"
    output_dir = sample_directory(output_root, subset_name)
    positive_ids = set().union(*relevant_docs.values())
    metadata = {
        **source_metadata,
        "subset_name": subset_name,
        "question_type_filter": type_label,
        "documents_filter": documents,
        "n_eligible_queries": n_eligible,
        "n_queries": len(queries),
        "n_corpus_docs": len(corpus),
        "n_positive_documents": len(positive_ids),
        "n_positive_relations": sum(len(ids) for ids in relevant_docs.values()),
        "seed": seed,
        "query_type_counts": dict(sorted(Counter(query_types.values()).items())),
        "query_types": query_types,
        "calibration_set_path": (
            str(calibration_set_path) if calibration_set_path else None
        ),
        "n_calibration_documents": len(calibration_document_ids or ()),
        "sampling": "filtered expert questions plus all gold documents and uniformly sampled corpus negatives",
    }
    save_sample(output_dir, queries, relevant_docs, corpus, metadata)
    dataset_id = register_dataset(output_dir)
    logger.info("Dataset registered as %s", dataset_id)
    return queries, relevant_docs, corpus, metadata, output_dir, dataset_id


## Create `list` subsets

This block creates `list-one` and `list-multiple`.

In [ ]:
created_subsets = globals().get("created_subsets", {})
for document_filter in ("one", "multiple"):
    key = f"list-{document_filter}"
    created_subsets[key] = create_bioasq_sample(
        question_type="list",
        documents=document_filter,
        subset_name=key,
        benchmark=benchmark,
        calibration_document_ids=calibration_document_ids,
        calibration_set_path=calibration_output_dir,
    )
    print(
        f"Created {key}: {created_subsets[key][4]} "
        f"({created_subsets[key][5]})"
    )


## Create `factoid` subsets

This block creates `factoid-one` and `factoid-multiple`.

In [ ]:
created_subsets = globals().get("created_subsets", {})
for document_filter in ("one", "multiple"):
    key = f"factoid-{document_filter}"
    created_subsets[key] = create_bioasq_sample(
        question_type="factoid",
        documents=document_filter,
        subset_name=key,
        benchmark=benchmark,
        calibration_document_ids=calibration_document_ids,
        calibration_set_path=calibration_output_dir,
    )
    print(
        f"Created {key}: {created_subsets[key][4]} "
        f"({created_subsets[key][5]})"
    )


## Create `summary` subsets

This block creates `summary-one` and `summary-multiple`.

In [ ]:
created_subsets = globals().get("created_subsets", {})
for document_filter in ("one", "multiple"):
    key = f"summary-{document_filter}"
    created_subsets[key] = create_bioasq_sample(
        question_type="summary",
        documents=document_filter,
        subset_name=key,
        benchmark=benchmark,
        calibration_document_ids=calibration_document_ids,
        calibration_set_path=calibration_output_dir,
    )
    print(
        f"Created {key}: {created_subsets[key][4]} "
        f"({created_subsets[key][5]})"
    )


## Browse subset examples

Change `EXAMPLE_SUBSET` and `EXAMPLE_PAGE`, then rerun the cell. The selected subset must have been created in this runtime.

In [ ]:
from IPython.display import Markdown, display

EXAMPLE_SUBSET = "list-multiple"
EXAMPLE_PAGE = 0
EXAMPLES_PER_PAGE = 3

if EXAMPLE_SUBSET not in created_subsets:
    raise KeyError(f"Create {EXAMPLE_SUBSET!r} before browsing it")
(
    queries,
    relevant_docs,
    corpus,
    metadata,
    output_dir,
    dataset_id,
) = created_subsets[EXAMPLE_SUBSET]
query_items = list(queries.items())
n_pages = max(1, (len(query_items) + EXAMPLES_PER_PAGE - 1) // EXAMPLES_PER_PAGE)
page = EXAMPLE_PAGE % n_pages
start = page * EXAMPLES_PER_PAGE

display(Markdown(
    f"### {EXAMPLE_SUBSET}: page {page + 1} of {n_pages}  "
    f"\nEligible questions: {metadata['n_eligible_queries']}"
))
for qid, query in query_items[start:start + EXAMPLES_PER_PAGE]:
    snippets = []
    for doc_id in sorted(relevant_docs[qid]):
        text = corpus[doc_id].replace("\n", " ")
        suffix = "…" if len(text) > 700 else ""
        snippets.append(f"**Document `{doc_id}`:** {text[:700]}{suffix}")
    display(Markdown(
        f"**Query `{qid}` ({metadata['query_types'][qid]}):** {query}\n\n"
        + "\n\n".join(snippets)
    ))
